In [1]:
import sys

sys.path.append("..")

import numpy as np
from classes.loader.loader import Loader
from classes.signal import SignalTime
from classes.common import Cables
import matplotlib.pyplot as plt
import scipy
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
from scipy.signal import hilbert, butter, sosfilt
import pywt

c:\Users\eduar\AppData\Local\pypoetry\Cache\virtualenvs\doctorate-VzOapWKR-py3.9\lib\site-packages\pydantic\_internal\_fields.py:149: UserWarning: Field "model_server_url" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
c:\Users\eduar\AppData\Local\pypoetry\Cache\virtualenvs\doctorate-VzOapWKR-py3.9\lib\site-packages\pydantic\_internal\_config.py:321: UserWarning: Valid config keys have changed in V2:
* 'schema_extra' has been renamed to 'json_schema_extra'
  warnings.warn(message, UserWarning)


In [2]:
loader = Loader(r"..\raw\sequence_1")
sample = loader.load_one("2022_11_19_08_34.json")
sample.convert_to_time()

In [3]:
signal = sample[0].y

In [4]:
def cosine_mod(signal: SignalTime, f_hz: float) -> SignalTime:
    t, x = signal.get()
    t = np.array(t)
    return SignalTime(np.array(x)*np.cos(2*np.pi*f_hz*t) * 0.5, signal.ts, signal.id)

In [5]:
def exp_forward_mod(signal: SignalTime, f_hz: float) -> SignalTime:
    t, x = signal.get()
    t = np.array(t)
    return SignalTime(np.array(x)*np.exp(-1j*2*np.pi*f_hz*t), signal.ts, signal.id)

def exp_backward_mod(signal: SignalTime, f_hz: float) -> SignalTime:
    t, x = signal.get()
    t = np.array(t)
    return SignalTime(np.array(x)*np.exp(1j*2*np.pi*f_hz*t), signal.ts, signal.id)

In [6]:
def decimate(signal: SignalTime, factor: int) -> SignalTime:
    t, x = signal.get()
    t = np.array(t)
    x = np.array(x)
    ind = np.arange(0, len(x), factor)
    new_t = t[ind]
    new_x = x[ind]
    return SignalTime(new_x, new_t[1]-new_t[0], signal.id)

In [7]:
def apply_filter(signal: SignalTime, filter) -> SignalTime:
    filtered = sosfilt(filter, signal.signal)
    return  SignalTime(filtered, signal.ts, signal.id)

In [39]:
class FilterBank:
    def __init__(self, num_filters: int, fs: float, filter_len: int):
        self.fs = fs
        self.num_filters = num_filters
        self.bandwidth = self.fs / 2 / self.num_filters
        self.filter_len = filter_len

    @property
    def base_lp_filter(self):
        return butter(
            N=self.filter_len,
            Wn=self.bandwidth,
            btype="lowpass",
            fs=self.fs,
            output="sos",
        )

    def transform(self, signal: SignalTime) -> list[SignalTime]:
        signals = []
        for k in range(self.num_filters):
            mod_f = apply_filter(
                exp_forward_mod(signal, self.bandwidth * (k + 1)), self.base_lp_filter
            )
            mod_b = apply_filter(
                exp_backward_mod(signal, self.bandwidth * (k + 1)), self.base_lp_filter
            )
            filtered = SignalTime(
                np.array(mod_f.signal) + np.array(mod_b.signal), signal.ts, signal.id
            )
            decimated = decimate(filtered, self.num_filters)
            decimated.signal = np.real(decimated.signal)
            signals.append(decimated)
        return signals

In [47]:
fb = FilterBank(num_filters=16, fs=1/signal.ts, filter_len=50)

In [48]:
signals = fb.transform(signal)

In [49]:
s = signal.to_dft_spectrum()
px.line(x=s.frequencies, y=s.magnitudes)

In [58]:
i = 3
s = signals[i].to_rspectrum(window="hamming", nperseg=256, poverlap=0.5)
px.line(x=np.array(s.frequencies) + i * fb.bandwidth, y=s.magnitudes)